# Project: Unemployment Analysis in India (Pre & Post Covid-19)

## Dataset Overview
The `Unemployment in India.csv` dataset contains monthly economic data spanning from **May 2019 to June 2020**. This timeframe is highly valuable because it captures the transition from a stable pre-pandemic economy to the immediate, severe economic shock caused by the Covid-19 lockdowns in early 2020. 

**Key Metrics Tracked:**
* **Region:** The specific Indian State.
* **Date:** The monthly observation date.
* **Estimated Unemployment Rate (%):** The percentage of the labor force that is unemployed.
* **Estimated Employed:** The absolute number of employed individuals.
* **Estimated Labour Participation Rate (%):** The percentage of the working-age population participating in the labor force.
* **Area:** Categorization of the region into **Rural** or **Urban**.

## 📑 Project Index (Step-by-Step Journey)
1. **Environment Setup & Data Loading:** Importing Pandas, Matplotlib, and Seaborn, and loading the CSV file.
2. **Data Cleaning & Preprocessing:** Removing missing values, stripping hidden spaces from text, and converting text dates into intelligent chronological objects.
3. **Exploratory Data Analysis (EDA):** Generating summary statistics to understand the basic mathematical distribution of the data.
4. **Time Series Analysis (The Covid-19 Impact):** Visualizing how the unemployment rate changed over time, specifically comparing Rural vs. Urban areas.
5. **Regional Analysis:** Using bar charts to identify which specific states suffered the highest average unemployment during this period.

## Step 1: Environment Setup and Data Loading
Every data analysis project starts by importing the necessary libraries and loading the raw data into the environment. 
* **`pandas` (pd):** The standard library for data manipulation. We use it to load our CSV file into a structural format called a DataFrame.
* **`matplotlib.pyplot` (plt):** The core plotting engine in Python.
* **`seaborn` (sns):** A data visualization library built on top of matplotlib that provides a high-level interface for drawing attractive statistical graphics.

We will use `pd.read_csv()` to read the file and `.head()` to inspect the first five rows, ensuring the data loaded correctly.

In [16]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Set a clean visual theme for our future charts
sns.set_theme(style="whitegrid")

# Load the dataset into a Pandas DataFrame
df_india = pd.read_csv('../data/raw/Unemployment in India.csv')

# Display the first 5 rows of the DataFrame
df_india.head()

,Region,Date,Frequency,Estimated Unemployment Rate (%),Estimated Employed,Estimated Labour Participation Rate (%),Area
0,Andhra Pradesh,31-05-2019,Monthly,3.65,11999139.0,43.24,Rural
1,Andhra Pradesh,30-06-2019,Monthly,3.05,11755881.0,42.05,Rural
2,Andhra Pradesh,31-07-2019,Monthly,3.75,12086707.0,43.50,Rural
3,Andhra Pradesh,31-08-2019,Monthly,3.32,12285693.0,43.97,Rural
4,Andhra Pradesh,30-09-2019,Monthly,5.17,12256762.0,44.68,Rural


## Step 2: Data Cleaning & Preprocessing
Real-world datasets often contain formatting artifacts. In this file, there are several completely blank rows at the bottom of the dataset.

1. **Clean Column Names:** We use `.columns.str.strip()` to remove hidden leading/trailing spaces (e.g., changing `' Date'` to `'Date'`).
2. **Handle Missing Values:** We use `.dropna(how='all')` to target and remove any rows where *every single column* is missing (NaN).
3. **Clean Text Data:** We apply `.str.strip()` to the 'Date', 'Region', and 'Area' columns to remove hidden spaces inside the data cells.
4. **Convert to Datetime:** We use `pd.to_datetime()` to convert raw text dates (like "31-05-2019") into intelligent chronological objects, allowing us to sort the data properly.

In [17]:
# 1. Clean the column names (remove leading/trailing spaces)
df_india.columns = df_india.columns.str.strip()

# 2 . Check for missing values in the dataset
missing_values = df_india.isnull().sum()

# 3. Display the number of missing values for each column
print("Missing values in each column:")
print(missing_values)

# 4. Drop rows with missing values (if any)
df_india = df_india.dropna(how='all').copy()

# 5. Clean the actual string data inside the categorical columns
df_india['Date'] = df_india['Date'].str.strip()
df_india['Region'] = df_india['Region'].str.strip()
df_india['Area'] = df_india['Area'].str.strip()

#6. Convert the 'Date' column to datetime format
df_india['Date'] = pd.to_datetime(df_india['Date'] , format='%d-%m-%Y')

# 7. Sort the entire dataset chronologically
df_india = df_india.sort_values(by='Date')

# 8. Display the cleaned DataFrame
df_india.info()

Missing values in each column:
Region                                     29
Date                                       29
Frequency                                  29
Estimated Unemployment Rate (%)            29
Estimated Employed                         29
Estimated Labour Participation Rate (%)    29
Area                                       29
dtype: int64
<class 'pandas.DataFrame'>
Index: 740 entries, 0 to 753
Data columns (total 7 columns):
 #   Column                                   Non-Null Count  Dtype         
---  ------                                   --------------  -----         
 0   Region                                   740 non-null    str           
 1   Date                                     740 non-null    datetime64[us]
 2   Frequency                                740 non-null    str           
 3   Estimated Unemployment Rate (%)          740 non-null    float64       
 4   Estimated Employed                       740 non-null    float64       
 5   Es

## Step 3: Exploratory Data Analysis (EDA) - Summary Statistics
To understand the baseline of our economic data, we will generate summary statistics for our numerical columns using the `.describe()` function.

This function automatically calculates:
* **Count:** The number of valid rows (which should be 740 for all columns now).
* **Mean:** The average value.
* **Min/Max:** The absolute lowest and highest values recorded.
* **25%, 50%, 75%:** The percentiles (the 50% percentile is the median).

We will chain `.round(2)` to the end to keep our percentages easy to read at two decimal places.

In [18]:
summary = df_india.describe().round(2)
summary

,Date,Estimated Unemployment Rate (%),Estimated Employed,Estimated Labour Participation Rate (%)
count,740,740.00,740.00,740.00
mean,2019-12-12 18:36:58.378378,11.79,7204460.03,42.63
min,2019-05-31 00:00:00,0.00,49420.00,13.33
25%,2019-08-31 00:00:00,4.66,1190404.50,38.06
50%,2019-11-30 00:00:00,8.35,4744178.50,41.16
75%,2020-03-31 00:00:00,15.89,11275489.50,45.50
max,2020-06-30 00:00:00,76.74,45777509.00,72.57
std,NaN,10.72,8087988.43,8.11


## Step 4: Interactive Time Series Analysis (The Covid-19 Impact)
To create a clean time-series chart in Plotly, we first need to mathematically aggregate our data. 

1. **`groupby(['Date', 'Area'])`:** We group our dataset by both the specific date and whether it is Rural/Urban.
2. **`.mean().reset_index()`:** We calculate the average unemployment rate for those specific buckets and flatten the table.
3. **`px.line()`:** We plot this newly aggregated, clean data to show the true average trend over time.

In [19]:
# 1. Manually calculate the average unemployment rate per Date and Area
df_trend = df_india.groupby(['Date', 'Area'])['Estimated Unemployment Rate (%)'].mean().reset_index()

# 2. Draw the interactive line chart using the aggregated data
fig = px.line(
    df_trend,                                # Use the newly grouped data, NOT the raw df_india
    x='Date',
    y='Estimated Unemployment Rate (%)',
    color='Area',
    markers=True,
    title='Interactive Unemployment Rate in India (May 2019 - June 2020) by Area'
)

# 3. Update the layout for the unified hover effect
fig.update_layout(
    xaxis_title='Date',
    yaxis_title='Estimated Unemployment Rate (%)',
    hovermode='x unified'
)

# 4. Display the chart
fig.show()

## Step 5: Interactive Regional Analysis (Identifying Hardest-Hit States)
To figure out which states suffered the most overall, we need to calculate the average unemployment rate for each specific region across the entire dataset. We will use an interactive horizontal bar chart so we can easily read the state names and hover for exact numbers.

1. **`groupby('Region')`**: Groups the data by state.
2. **`.mean()`**: Calculates the average of the numerical columns for each state.
3. **`px.bar()`**: Plotly Express's function to generate the bar chart.
4. **`color='Estimated Unemployment Rate (%)'`**: This automatically applies a color gradient (heatmap style) to the bars based on how high the unemployment rate is.

In [28]:
# 1. Group the data by Region and calculate the average unemployment rate
regional_unemployment = df_india.groupby('Region')['Estimated Unemployment Rate (%)'].mean().reset_index()

# 2. Sort the values from highest to lowest (ascending=False)
regional_unemployment = regional_unemployment.sort_values(by='Estimated Unemployment Rate (%)')

# 3. Create the interactive horizontal bar chart
fig = px.bar(
    regional_unemployment, 
    x='Estimated Unemployment Rate (%)', 
    y='Region', 
    orientation='h',  # 'h' forces a horizontal bar chart
    color='Estimated Unemployment Rate (%)', # Map colors to the unemployment value
    color_continuous_scale='Reds',           # Use a red color scale (darker red = worse unemployment)
    title='Interactive Average Unemployment Rate by State (May 2019 - June 2020)',
    height=800  # Make the chart tall enough to comfortably fit all 28 states
)

# 4. Display the interactive chart
fig.show()

## Step 6: Interactive Correlation Matrix (Understanding Metric Relationships)
To understand how our numerical variables interact with each other (for example, does a high labour participation rate mean higher employment?), we will calculate the mathematical correlation and visualize it using an interactive heatmap.

In [29]:
# 1. Select only the numerical columns for correlation
numerical_cols = df_india[['Estimated Unemployment Rate (%)', 'Estimated Employed', 'Estimated Labour Participation Rate (%)']]

# 2. Calculate the Pearson correlation matrix
corr_matrix = numerical_cols.corr()

# 3. Plot the interactive heatmap using Plotly's imshow
fig = px.imshow(
    corr_matrix,
    text_auto=True,                  # Automatically display the correlation numbers
    aspect="auto",                   # Adjusts the cell sizes to fit the screen
    color_continuous_scale='RdBu_r', # Red-Blue color scale (standard for correlations)
    title='Interactive Correlation Matrix of Employment Metrics'
)

# 4. Display the chart
fig.show()

## Step 7: Interactive Outlier Detection (Analyzing Extreme Shocks)
During the COVID-19 pandemic, certain regions experienced sudden, massive spikes in unemployment. To properly identify these extreme events without losing the context of where and when they happened, we will use an interactive boxplot.

1. **`px.box()`**: The standard Plotly function for drawing box-and-whisker plots to show data distribution (median, quartiles, and outliers).
2. **`hover_data=['Region', 'Date']`**: This powerful parameter injects extra columns from our dataframe into the chart's tooltips. By hovering over any outlier dot, we can instantly identify which state experienced the shock and on what date, completely eliminating the need to write separate query code to find these anomalies.

In [30]:
# 1. Create the interactive boxplot
fig = px.box(
    df_india,
    x='Area',
    y='Estimated Unemployment Rate (%)',
    color='Area',
    hover_data=['Region', 'Date'],  # Injecting Region and Date for the hover tooltips
    title='Interactive Distribution and Outliers of Unemployment Rates by Area'
)

# 2. Update layout for clean axis labels
fig.update_layout(
    xaxis_title='Area',
    yaxis_title='Estimated Unemployment Rate (%)'
)

# 3. Display the chart
fig.show()

## Step 8: Feature Engineering (Pre-Covid vs. Covid-19 Impact)
To quantify the exact impact of the pandemic, we will engineer a new feature called `Period`. We know the major lockdowns began in late March 2020. By splitting our dataset into "Pre-Covid" (before April 2020) and "During Covid" (April 2020 onwards), we can directly compare the averages.

In [31]:
# 1. Create a new column 'Period' based on the Date
# We use a lambda function to check if the month is before April 2020 (Month 4 of 2020)
df_india['Period'] = df_india['Date'].apply(lambda x: 'Pre-Covid' if x < pd.Timestamp('2020-04-01') else 'During Covid')

# 2. Group the data by Region and Period to get the mean unemployment rate
period_impact = df_india.groupby(['Region', 'Period'])['Estimated Unemployment Rate (%)'].mean().reset_index()

# 3. Create an interactive grouped bar chart
fig = px.bar(
    period_impact,
    x='Region',
    y='Estimated Unemployment Rate (%)',
    color='Period',
    barmode='group', # Puts the Pre-Covid and Covid bars side-by-side for each state
    title='Interactive Comparison: Pre-Covid vs. During Covid Unemployment Rate by State',
    color_discrete_map={'Pre-Covid': 'royalblue', 'During Covid': 'crimson'} # Custom colors
)

fig.update_layout(xaxis_tickangle=-45)
fig.show()

## Phase 2: Geospatial Analysis of Unemployment
To visualize the geographic spread of the economic crisis, we will import our second dataset, `Unemployment_Rate_upto_11_2020.csv`. This dataset includes Longitude and Latitude coordinates for each state, allowing us to map the data directly onto an interactive map of India.

1. **`pd.read_csv()`**: Load the new dataset.
2. **Column Cleaning**: Just like the first dataset, we will strip any trailing spaces from the column names to avoid `KeyError` issues later.

In [32]:
# 1. Load the second dataset
df_geo = pd.read_csv('../data/raw/Unemployment_Rate_upto_11_2020.csv')

# 2. Clean the column names by removing extra spaces
df_geo.columns = df_geo.columns.str.strip()

# 3. Display the first few rows to verify the new Longitude and Latitude columns
df_geo.head()

,Region,Date,Frequency,Estimated Unemployment Rate (%),Estimated Employed,Estimated Labour Participation Rate (%),Region.1,longitude,latitude
0,Andhra Pradesh,31-01-2020,M,5.48,16635535,41.02,South,15.9129,79.74
1,Andhra Pradesh,29-02-2020,M,5.83,16545652,40.90,South,15.9129,79.74
2,Andhra Pradesh,31-03-2020,M,5.79,15881197,39.18,South,15.9129,79.74
3,Andhra Pradesh,30-04-2020,M,20.51,11336911,33.10,South,15.9129,79.74
4,Andhra Pradesh,31-05-2020,M,17.43,12988845,36.46,South,15.9129,79.74
